In [ ]:
#!/usr/bin/env python
"""
Notebook: 05_error_analysis.ipynb
Model Error Analysis and Interpretation
"""

# Model Error Analysis
 
# This notebook analyzes model predictions, errors, and provides interpretability:
 - Confusion matrix analysis
 - Error case examination
 - Feature importance
 - Grad-CAM visualization

# 1. Setup and Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.manifold import TSNE
import sys

sys.path.insert(0, '..')

from src.models.cnn_classifier import ECG1DCNN
from src.training.metrics import MetricsCalculator

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# 2. Load Model and Data

In [ ]:
# Load trained model (using demo data)
input_dim = 187
num_classes = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = ECG1DCNN(input_dim=input_dim, num_classes=num_classes).to(device)

# Generate test data
np.random.seed(123)
num_test = 200
X_test = np.random.randn(num_test, 1, input_dim) * 0.1
y_test = np.random.randint(0, num_classes, num_test)

# Add some class-specific patterns
for i, class_id in enumerate(y_test):
    if class_id == 0:  # Normal
        X_test[i, 0, 80:110] += 1.0 * np.hanning(30)
    elif class_id == 1:  # AFib
        X_test[i, 0, 80:110] += 0.8 * np.hanning(30)
    elif class_id == 2:  # PVC
        X_test[i, 0, 70:120] += 1.2 * np.hanning(50)

# Get predictions
model.eval()
with torch.no_grad():
    logits = model(torch.FloatTensor(X_test).to(device))
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    y_pred = np.argmax(probs, axis=1)

class_names = ['Normal', 'AFib', 'PVC', 'Bradycardia', 'Other']

# 3. Confusion Matrix Analysis

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Per-class metrics
calculator = MetricsCalculator(num_classes=num_classes, class_names=class_names)
report = calculator.compute_classification_report(y_test, y_pred)

print("\nClassification Report:")
print("=" * 60)
for class_name in class_names:
    print(f"\n{class_name}:")
    print(f"  Precision: {report[class_name]['precision']:.3f}")
    print(f"  Recall: {report[class_name]['recall']:.3f}")
    print(f"  F1-Score: {report[class_name]['f1-score']:.3f}")
    print(f"  Support: {report[class_name]['support']}")

# 4. Error Analysis by Class

In [ ]:
# Identify misclassified samples
errors = []
for i, (true, pred) in enumerate(zip(y_test, y_pred)):
    if true != pred:
        errors.append({
            'index': i,
            'true_class': class_names[true],
            'pred_class': class_names[pred],
            'confidence': probs[i][pred],
            'true_confidence': probs[i][true]
        })

error_df = pd.DataFrame(errors)
print(f"Total errors: {len(errors)}/{num_test} ({len(errors)/num_test*100:.1f}%)")
print("\nError distribution:")
print(error_df.groupby(['true_class', 'pred_class']).size())

# Plot error confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(error_df['confidence'], bins=20, color='red', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Prediction Confidence')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Confidence of Misclassified Samples')
axes[0].grid(True, alpha=0.3)

axes[1].hist(error_df['true_confidence'], bins=20, color='green', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('True Class Probability')
axes[1].set_ylabel('Frequency')
axes[1].set_title('True Class Probability of Errors')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. Visualize Error Cases

In [ ]:
# Show some example misclassifications
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
axes = axes.flatten()

for idx, error in enumerate(errors[:6]):
    sample_idx = error['index']
    true_class = error['true_class']
    pred_class = error['pred_class']
    confidence = error['confidence']
    
    time = np.arange(input_dim) / 360 * 1000
    axes[idx].plot(time, X_test[sample_idx, 0], 'b-', linewidth=1.5)
    axes[idx].set_title(f'True: {true_class} | Pred: {pred_class} (conf={confidence:.2f})')
    axes[idx].set_xlabel('Time (ms)')
    axes[idx].set_ylabel('Amplitude (mV)')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. Feature Space Visualization with t-SNE

In [ ]:
# Extract features from the model
def extract_features(model, X):
    model.eval()
    features = []
    with torch.no_grad():
        for i in range(0, len(X), 32):
            batch = torch.FloatTensor(X[i:i+32]).to(device)
            # Get features before classifier
            x = model.conv_block1(batch)
            x = model.conv_block2(x)
            x = model.conv_block3(x)
            x = model.global_avg_pool(x)
            x = x.view(x.size(0), -1)
            features.append(x.cpu().numpy())
    return np.concatenate(features, axis=0)

features = extract_features(model, X_test)

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(features)

# Plot t-SNE visualization
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(features_2d[:, 0], features_2d[:, 1], 
                    c=y_test, cmap='tab10', s=50, alpha=0.7)

# Add labels for misclassified points
for i, error in enumerate(errors):
    if i < 20:  # Show first 20 errors
        ax.annotate('X', (features_2d[error['index'], 0], features_2d[error['index'], 1]),
                   fontsize=12, color='red', fontweight='bold')

ax.set_xlabel('t-SNE Component 1')
ax.set_ylabel('t-SNE Component 2')
ax.set_title('t-SNE Visualization of Learned Features')
plt.colorbar(scatter, ax=ax, ticks=range(num_classes), label='Class')
plt.tight_layout()
plt.show()

# 7. Confidence Calibration

In [ ]:
# Calculate calibration curve
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(10, 8))

for class_idx in range(num_classes):
    # Get predicted probabilities for this class
    prob_pos = probs[:, class_idx]
    y_true_binary = (y_test == class_idx).astype(int)
    
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_true_binary, prob_pos, n_bins=10, strategy='uniform'
    )
    
    ax.plot(mean_predicted_value, fraction_of_positives, 'o-', 
           label=class_names[class_idx], linewidth=2, markersize=8)

# Perfect calibration line
ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated', linewidth=2)

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves by Class')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 8. Confidence Distribution Analysis

In [ ]:
# Analyze confidence by correctness
correct_mask = (y_pred == y_test)
correct_conf = probs[np.arange(len(probs)), y_pred][correct_mask]
error_conf = probs[np.arange(len(probs)), y_pred][~correct_mask]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(correct_conf, bins=30, alpha=0.7, color='green', edgecolor='black', label='Correct')
axes[0].hist(error_conf, bins=30, alpha=0.7, color='red', edgecolor='black', label='Incorrect')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Confidence Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
bp_data = [correct_conf, error_conf]
axes[1].boxplot(bp_data, labels=['Correct', 'Incorrect'], patch_artist=True)
axes[1].set_ylabel('Confidence Score')
axes[1].set_title('Confidence Comparison')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Mean confidence - Correct: {np.mean(correct_conf):.3f}")
print(f"Mean confidence - Incorrect: {np.mean(error_conf):.3f}")